# Cotton benchmark — evaluation against the verified gold set

Scores each trained baseline on the 1{,}215 human-verified tiles. Produces
the main results table, per-field and per-density breakdowns, and counting
error. Runtime: GPU, ~15 min total.

Needs in Drive `cotton_bench/`: `gold_v2_verified.zip`, `gold_v2_tiles.zip`,
`test_classes.csv`, and `baselines/<model>/best.pt` for each model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics pycocotools
import os, json, glob
for f in ['gold_v2_verified.zip','gold_v2_tiles.zip','test_classes.csv']:
    print(f, 'OK' if os.path.exists(f'{DRIVE}/{f}') else 'MISSING')
MODELS = sorted(os.listdir(f'{DRIVE}/baselines'))
print('models:', MODELS)

In [ ]:
# Unpack gold tiles and verified annotations
!rm -rf /content/gold /content/gold_ann
!unzip -q {DRIVE}/gold_v2_tiles.zip -d /content/gold
!unzip -q {DRIVE}/gold_v2_verified.zip -d /content/gold_ann

tile_dir = '/content/gold/gold_v2'
tiles = sorted(glob.glob(f'{tile_dir}/*.jpg') + glob.glob(f'{tile_dir}/*.JPG'))
tiles = sorted(set(tiles))
ann_path = glob.glob('/content/gold_ann/**/instances_default.json', recursive=True)[0]
gold = json.load(open(ann_path))
print(len(tiles), 'tiles on disk |', len(gold['images']), 'images in COCO |',
      len(gold['annotations']), 'boxes')

In [ ]:
# Normalise gold to {tile_name: [[x1,y1,x2,y2], ...]} and attach metadata
import re, csv
from collections import defaultdict

name_of = {im['id']: im['file_name'].split('/')[-1] for im in gold['images']}
gt = defaultdict(list)
for a in gold['annotations']:
    x, y, w, h = a['bbox']
    gt[name_of[a['image_id']]].append([x, y, x + w, y + h])
gt = {t: v for t, v in gt.items()}
for t in name_of.values():          # keep genuinely empty tiles
    gt.setdefault(t, [])

cls_of_stem = {}
with open(f'{DRIVE}/test_classes.csv') as f:
    for row in csv.DictReader(f):
        cls_of_stem[row['stem']] = row['class']
def stem_of(n): return re.split(r'_r\d+_c\d+_x\d+_y\d+', n)[0]
field = {t: cls_of_stem.get(stem_of(t), '?') for t in gt}

def dbin(n):
    return ('0' if n == 0 else '1-50' if n <= 50 else '51-150' if n <= 150
            else '151-300' if n <= 300 else '300+')
bins = {t: dbin(len(v)) for t, v in gt.items()}

import numpy as np
counts = np.array([len(v) for v in gt.values()])
print(f'gold: {len(gt)} tiles, {counts.sum():,} boxes, '
      f'mean {counts.mean():.0f}, median {np.median(counts):.0f}, max {counts.max()}')
from collections import Counter
print('by field:', Counter(field.values()))
print('by density:', dict(sorted(Counter(bins.values()).items())))

## Matching and metrics

Greedy IoU matching by descending confidence, the standard COCO-style
assignment. AP is computed by 101-point interpolation over the
precision-recall curve.

In [ ]:
def iou_matrix(A, B):
    if len(A) == 0 or len(B) == 0:
        return np.zeros((len(A), len(B)), dtype=np.float32)
    A = np.asarray(A, np.float32); B = np.asarray(B, np.float32)
    ix1 = np.maximum(A[:,0,None], B[None,:,0]); iy1 = np.maximum(A[:,1,None], B[None,:,1])
    ix2 = np.minimum(A[:,2,None], B[None,:,2]); iy2 = np.minimum(A[:,3,None], B[None,:,3])
    inter = np.clip(ix2-ix1, 0, None) * np.clip(iy2-iy1, 0, None)
    aA = ((A[:,2]-A[:,0])*(A[:,3]-A[:,1]))[:,None]
    aB = ((B[:,2]-B[:,0])*(B[:,3]-B[:,1]))[None,:]
    union = aA + aB - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

def ap_at(preds, gtd, thr, tiles_subset=None):
    """preds: {tile: [[x1,y1,x2,y2,conf],...]}; returns AP, precision, recall."""
    keys = tiles_subset if tiles_subset is not None else list(gtd)
    rows, n_gt = [], 0
    for t in keys:
        g = gtd.get(t, []); p = sorted(preds.get(t, []), key=lambda b: -b[4])
        n_gt += len(g)
        if not p:
            continue
        M = iou_matrix([b[:4] for b in p], g)
        taken = set()
        for i, b in enumerate(p):
            j, best = -1, thr
            if M.shape[1]:
                order = np.argsort(-M[i])
                for cand in order:
                    if M[i, cand] < thr: break
                    if cand not in taken:
                        j, best = int(cand), M[i, cand]; break
            if j >= 0:
                taken.add(j); rows.append((b[4], 1))
            else:
                rows.append((b[4], 0))
    if n_gt == 0 or not rows:
        return 0.0, 0.0, 0.0
    rows.sort(key=lambda r: -r[0])
    tp = np.cumsum([r[1] for r in rows]); fp = np.cumsum([1-r[1] for r in rows])
    rec = tp / n_gt; prec = tp / np.maximum(tp + fp, 1e-9)
    # 101-point interpolated AP
    mprec = np.concatenate([[0], prec, [0]])
    for i in range(len(mprec)-2, -1, -1):
        mprec[i] = max(mprec[i], mprec[i+1])
    ap = float(np.mean(np.interp(np.linspace(0,1,101), np.concatenate([[0],rec,[1]]),
                                 mprec[:len(rec)+2])))
    return ap, float(prec[-1]), float(rec[-1])

def eval_subset(preds, gtd, keys):
    ap50, p, r = ap_at(preds, gtd, 0.5, keys)
    aps = [ap_at(preds, gtd, t, keys)[0] for t in np.arange(0.5, 1.0, 0.05)]
    return dict(mAP50=ap50, mAP50_95=float(np.mean(aps)), P=p, R=r)

In [ ]:
# Run every baseline over the gold tiles
from ultralytics import YOLO
from tqdm import tqdm

CONF = 0.001      # low threshold: AP integrates the full PR curve
all_preds = {}
for name in MODELS:
    w = f'{DRIVE}/baselines/{name}/best.pt'
    if not os.path.exists(w):
        print('skip', name); continue
    model = YOLO(w)
    preds = {}
    for i in tqdm(range(0, len(tiles), 64), desc=name):
        batch = tiles[i:i+64]
        res = model.predict(batch, imgsz=640, conf=CONF, iou=0.7,
                            max_det=1000, verbose=False)
        for path, r in zip(batch, res):
            b = r.boxes
            preds[os.path.basename(path)] = [
                [float(x1), float(y1), float(x2), float(y2), float(c)]
                for (x1,y1,x2,y2), c in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy())]
    all_preds[name] = preds
    n = sum(len(v) for v in preds.values())
    print(f'{name}: {n:,} predictions ({n/len(preds):.0f} per tile)')

json.dump({k: {t: [[round(x,2) for x in b] for b in v] for t, v in p.items()}
           for k, p in all_preds.items()},
          open(f'{DRIVE}/gold_predictions.json','w'))
print('predictions saved to Drive')

## Main results

In [ ]:
import pandas as pd
keys = list(gt)
rows = []
for name, preds in all_preds.items():
    m = eval_subset(preds, gt, keys)
    rows.append(dict(model=name, **{k: round(v, 4) for k, v in m.items()}))
main = pd.DataFrame(rows).set_index('model')
print(main.to_string())
main.to_csv(f'{DRIVE}/results_main.csv')

In [ ]:
# Per field
rows = []
for name, preds in all_preds.items():
    for f in ['predefoliation', 'postdefoliation']:
        sub = [t for t in keys if field[t] == f]
        m = eval_subset(preds, gt, sub)
        rows.append(dict(model=name, field=f, n_tiles=len(sub),
                         **{k: round(v, 4) for k, v in m.items()}))
byfield = pd.DataFrame(rows)
print(byfield.to_string(index=False))
byfield.to_csv(f'{DRIVE}/results_by_field.csv', index=False)

In [ ]:
# Per density bin
order = ['0','1-50','51-150','151-300','300+']
rows = []
for name, preds in all_preds.items():
    for b in order:
        sub = [t for t in keys if bins[t] == b]
        if not sub: continue
        m = eval_subset(preds, gt, sub)
        rows.append(dict(model=name, density=b, n_tiles=len(sub),
                         **{k: round(v, 4) for k, v in m.items()}))
bydens = pd.DataFrame(rows)
print(bydens.to_string(index=False))
bydens.to_csv(f'{DRIVE}/results_by_density.csv', index=False)

In [ ]:
# Counting error at the operating point that maximises F1 per model
rows = []
for name, preds in all_preds.items():
    best = None
    for thr in np.arange(0.05, 0.85, 0.05):
        err, rel = [], []
        for t in keys:
            pc = sum(1 for b in preds.get(t, []) if b[4] >= thr)
            gc = len(gt[t])
            err.append(abs(pc - gc))
            if gc: rel.append(abs(pc - gc) / gc)
        mae = float(np.mean(err))
        if best is None or mae < best['MAE']:
            best = dict(model=name, conf=round(float(thr),2), MAE=round(mae,2),
                        rel_err=round(float(np.mean(rel))*100, 1))
    rows.append(best)
counting = pd.DataFrame(rows)
print(counting.to_string(index=False))
counting.to_csv(f'{DRIVE}/results_counting.csv', index=False)